In [9]:
import pandas as pd

# funkcja do wczytania danych miejsko-powietrznych
def load_city_data(path, city_name):
    df = pd.read_excel(path, header=[0, 1, 2, 3])
    cols = []
    for i, col in enumerate(df.columns):
        if i == 0:
            cols.append("date")
            continue
        name = col[-1]
        if isinstance(name, str) and name.startswith("Unnamed"):
            for x in reversed(col):
                if isinstance(x, str) and not x.startswith("Unnamed"):
                    name = x
                    break
        cols.append(name)
    df.columns = cols
    df = df[df["date"].astype(str) != "Data"].copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["city"] = city_name
    df["disease"] = "J45"
    for col in df.columns:
        if col not in ["date", "city", "disease"]:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

# ścieżki
gliwice_path = "/workspaces/StatystykaProj/Miasta/Gliwice 24h 2007-2014.xlsx"
tychy_path = "/workspaces/StatystykaProj/Miasta/Tychy 24h 2007-2014.xlsx"

# wczytanie
gliwice = load_city_data(gliwice_path, "Gliwice")
tychy = load_city_data(tychy_path, "Tychy")

# podgląd
print("Gliwice:")
print(gliwice.head())

print("\nTychy:")
print(tychy.head())



Gliwice:
        date  NO        NO2  NOx       PM10  PM2.5        SO2  cisnienie  \
1 2007-01-01 NaN   9.521739  NaN  14.666667    NaN   4.000000        NaN   
2 2007-01-02 NaN  17.347826  NaN  22.291667    NaN        NaN        NaN   
3 2007-01-03 NaN  17.476190  NaN  17.217391    NaN   2.750000        NaN   
4 2007-01-04 NaN  20.047619  NaN  13.333333    NaN  10.952381        NaN   
5 2007-01-05 NaN  13.565217  NaN  13.791667    NaN   3.260870        NaN   

   kier  opad_atm  predk  prom  temp  wilg     city disease  
1   NaN       NaN    NaN   NaN   NaN   NaN  Gliwice     J45  
2   NaN       NaN    NaN   NaN   NaN   NaN  Gliwice     J45  
3   NaN       NaN    NaN   NaN   NaN   NaN  Gliwice     J45  
4   NaN       NaN    NaN   NaN   NaN   NaN  Gliwice     J45  
5   NaN       NaN    NaN   NaN   NaN   NaN  Gliwice     J45  

Tychy:
        date  NO        NO2  NOx         O3       PM10        SO2  cisnienie  \
1 2007-01-01 NaN  11.217391  NaN  50.416667  10.125000   9.608696        N

In [10]:
import re

path = "/workspaces/StatystykaProj/choroby/J45.sta"
with open(path, "rb") as f:
    text = f.read().decode("utf-8", errors="replace")

match = re.search(r"(SELECT .*?)(?=&|$)", text, re.S | re.I)
j45_query = match.group(1).strip() if match else None
print("J45 query extracted from .sta file:\n")
print(j45_query)

# tworzymy zbiorczy dataset dla obu miast
data = pd.concat([gliwice, tychy], ignore_index=True)
print("\nCombined J45 dataset shape:", data.shape)
data.to_csv("/workspaces/StatystykaProj/J45_cities_dataset.csv", index=False)
print("Saved combined dataset to /workspaces/StatystykaProj/J45_cities_dataset.csv")
print(data.head())

J45 query extracted from .sta file:

SELECT TABLE1.`DATA`,SUM(TABLE1.`LB_OSOB`) FROM `SUM_2007_2014` TABLE1 
WHERE (TABLE1.`GMINA_PACJENTA` = 'Gliwice' AND TABLE1.`ROZPOZNANIE` IN ('J45', 'J45.0', 'J45.1', 'J45.8', 'J45.9')) GROUP BY TABLE1.`DATA`

Combined J45 dataset shape: (5793, 17)
Saved combined dataset to /workspaces/StatystykaProj/J45_cities_dataset.csv
        date  NO        NO2  NOx       PM10  PM2.5        SO2  cisnienie  \
0 2007-01-01 NaN   9.521739  NaN  14.666667    NaN   4.000000        NaN   
1 2007-01-02 NaN  17.347826  NaN  22.291667    NaN        NaN        NaN   
2 2007-01-03 NaN  17.476190  NaN  17.217391    NaN   2.750000        NaN   
3 2007-01-04 NaN  20.047619  NaN  13.333333    NaN  10.952381        NaN   
4 2007-01-05 NaN  13.565217  NaN  13.791667    NaN   3.260870        NaN   

   kier  opad_atm  predk  prom  temp  wilg     city disease  O3  
0   NaN       NaN    NaN   NaN   NaN   NaN  Gliwice     J45 NaN  
1   NaN       NaN    NaN   NaN   NaN   NaN  Gli

In [11]:
print(data.head(10))

        date  NO        NO2  NOx       PM10  PM2.5        SO2  cisnienie  \
0 2007-01-01 NaN   9.521739  NaN  14.666667    NaN   4.000000        NaN   
1 2007-01-02 NaN  17.347826  NaN  22.291667    NaN        NaN        NaN   
2 2007-01-03 NaN  17.476190  NaN  17.217391    NaN   2.750000        NaN   
3 2007-01-04 NaN  20.047619  NaN  13.333333    NaN  10.952381        NaN   
4 2007-01-05 NaN  13.565217  NaN  13.791667    NaN   3.260870        NaN   
5 2007-01-06 NaN  14.000000  NaN  19.458333    NaN   3.695652        NaN   
6 2007-01-07 NaN  13.086957  NaN  13.625000    NaN   3.913043        NaN   
7 2007-01-08 NaN  31.782609  NaN  35.333333    NaN  13.782609        NaN   
8 2007-01-09 NaN  19.391304  NaN  23.375000    NaN   5.434783        NaN   
9 2007-01-10 NaN  16.173913  NaN  15.333333    NaN   3.238095        NaN   

   kier  opad_atm  predk  prom  temp  wilg     city disease  O3  
0   NaN       NaN    NaN   NaN   NaN   NaN  Gliwice     J45 NaN  
1   NaN       NaN    NaN   NaN 

In [12]:
print(data.columns.tolist())
print(data.info())

['date', 'NO', 'NO2', 'NOx', 'PM10', 'PM2.5', 'SO2', 'cisnienie', 'kier', 'opad_atm', 'predk', 'prom', 'temp', 'wilg', 'city', 'disease', 'O3']
<class 'pandas.DataFrame'>
RangeIndex: 5793 entries, 0 to 5792
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   date       5793 non-null   datetime64[us]
 1   NO         4521 non-null   float64       
 2   NO2        5500 non-null   float64       
 3   NOx        4935 non-null   float64       
 4   PM10       5213 non-null   float64       
 5   PM2.5      1763 non-null   float64       
 6   SO2        5569 non-null   float64       
 7   cisnienie  3249 non-null   float64       
 8   kier       2870 non-null   float64       
 9   opad_atm   758 non-null    float64       
 10  predk      2555 non-null   float64       
 11  prom       3143 non-null   float64       
 12  temp       2444 non-null   float64       
 13  wilg       2649 non-null   float64       
 14  cit

In [13]:
print(data.groupby(["city"]).size())
print(data[["city", "date", "NO", "NO2", "PM10", "PM2.5", "SO2"]].head())

city
Gliwice    2899
Tychy      2894
dtype: int64
      city       date  NO        NO2       PM10  PM2.5        SO2
0  Gliwice 2007-01-01 NaN   9.521739  14.666667    NaN   4.000000
1  Gliwice 2007-01-02 NaN  17.347826  22.291667    NaN        NaN
2  Gliwice 2007-01-03 NaN  17.476190  17.217391    NaN   2.750000
3  Gliwice 2007-01-04 NaN  20.047619  13.333333    NaN  10.952381
4  Gliwice 2007-01-05 NaN  13.565217  13.791667    NaN   3.260870
